# Lesson 21: Turning words into numbers — preprocessing, Bag of Words and TF-IDF 🔤➡️🔢
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Clean text with the classic **preprocessing** steps: tokenise, lowercase, remove stop words, stem / lemmatise
2. Represent documents as **Bag of Words** vectors
3. Calculate **TF-IDF** by hand, then with scikit-learn
4. Build a mini **search engine** for school notices

**⏱️ Timing:** preprocessing 15' | Bag of Words 10' | TF-IDF 15' | search engine 20'

> 💡 No GPU needed.

## 🧺 An analogy: sorting the shopping
Computers can't read; they only understand numbers. Before the 2010s (and still today for fast, simple jobs), NLP worked like unpacking groceries:
1. **Unpack** the bag into single items → *tokenising*
2. Make labels consistent ("Apples", "apples", "APPLES" → "apple") → *lowercasing, stemming, lemmatising*
3. Throw away the packaging ("the", "a", "is") → *removing stop words*
4. **Count** what's left → *Bag of Words*
5. Notice that rare items (saffron!) tell you more about a recipe than common ones (salt) → *TF-IDF*

In [ ]:
import nltk, re
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet"]:
    nltk.download(pkg, quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

text = "The students were running quickly to the libraries, because the libraries' Wi-Fi is FAST!!! 😄 Visit https://school.nz"
print(word_tokenize(text))

## Part 1: The preprocessing pipeline 🧹

In [ ]:
stop = set(stopwords.words("english"))
stemmer, lemmatizer = PorterStemmer(), WordNetLemmatizer()

def preprocess(text, show=False):
    steps = {}
    text = re.sub(r"https?://\S+", " ", text)                 # 1. remove links
    steps["no links"] = text
    text = text.lower()                                         # 2. lowercase
    steps["lowercase"] = text
    tokens = [t for t in word_tokenize(text) if t.isalpha()]   # 3. tokenise, keep only words
    steps["tokens"] = tokens
    tokens = [t for t in tokens if t not in stop]               # 4. remove stop words
    steps["no stop words"] = tokens
    lemmas = [lemmatizer.lemmatize(t, pos="v") for t in tokens] # 5. lemmatise (verbs → base form)
    lemmas = [lemmatizer.lemmatize(t) for t in lemmas]          #    (and nouns → singular)
    steps["lemmas"] = lemmas
    if show:
        for k, v in steps.items(): print(f"{k:14}: {v}")
    return lemmas

preprocess(text, show=True);

In [ ]:
# Stemming vs lemmatising: chopping vs looking it up in a dictionary
for w in ["running", "studies", "better", "libraries", "wolves", "happily", "was"]:
    print(f"{w:10} stem: {stemmer.stem(w):9} lemma: {lemmatizer.lemmatize(w, pos='v'):9} / {lemmatizer.lemmatize(w)}")

### 🧩 Exercise 1
1. Stemming makes non-words like `studi` and `happili`. Why might that still be fine for a search engine?
2. **Stop words can matter!** Preprocess `"This is not good"` and `"This is good"`. What went wrong? When should you *keep* stop words? (Sentiment, translation, chatbots…)
3. Modern LLMs skip most of these steps and use **subword tokenisers** instead (Lesson 5). Why? (They keep all the information and let the model learn what matters.)

## Part 2: Bag of Words 🎒
Each document becomes a row of counts, one column per word in the vocabulary. **Word order is thrown away** — that's why it's called a "bag".

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

docs = ["the dog chased the cat",
        "the cat sat on the mat",
        "the dog ate my homework"]
bow = CountVectorizer()
X = bow.fit_transform(docs)
pd.DataFrame(X.toarray(), columns=bow.get_feature_names_out(), index=["doc 1", "doc 2", "doc 3"])

In [ ]:
# The big weakness: different meaning, identical bag!
a, b = "the dog bit the man", "the man bit the dog"
v = CountVectorizer().fit([a, b])
print(v.transform([a]).toarray(), "\n", v.transform([b]).toarray(), "← identical")
# A partial fix: n-grams count pairs of words too
v2 = CountVectorizer(ngram_range=(1, 2)).fit([a, b])
print(v2.get_feature_names_out())
print(v2.transform([a]).toarray(), "\n", v2.transform([b]).toarray(), "← now different")

## Part 3: TF-IDF — rare words matter more 💎
- **TF (term frequency)**: how often a word appears **in this document**
- **IDF (inverse document frequency)**: how **rare** the word is across **all** documents: `idf = log(N / df)`
- **TF-IDF = TF × IDF** → high for words that are common *here* but rare *elsewhere*

"the" appears everywhere → IDF ≈ 0 → ignored. "homework" appears once → high IDF → important.

In [ ]:
import math
tokenised = [d.split() for d in docs]
N = len(docs)
vocab = sorted(set(w for d in tokenised for w in d))

def tf(word, doc):  return doc.count(word) / len(doc)
def idf(word):      return math.log(N / sum(word in d for d in tokenised))

table = {w: [round(tf(w, d) * idf(w), 3) for d in tokenised] for w in vocab}
print("IDF of 'the':", round(idf("the"), 3), "| IDF of 'homework':", round(idf("homework"), 3))
pd.DataFrame(table, index=["doc 1", "doc 2", "doc 3"])

In [ ]:
# scikit-learn's version (it adds smoothing and normalisation, so the numbers differ a little)
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
pd.DataFrame(tfidf.fit_transform(docs).toarray().round(2), columns=tfidf.get_feature_names_out(), index=["doc 1", "doc 2", "doc 3"])

### 🧩 Exercise 2
Which word has the **highest** TF-IDF score in each document? Does it describe what the document is about?

## Part 4: Build a search engine for school notices 🔍
We turn every notice **and** the search query into TF-IDF vectors, then rank notices by **cosine similarity** (how much two vectors point in the same direction: 1 = same, 0 = nothing in common).

In [ ]:
notices = [
    "The library will be closed on Monday for stocktaking. Books can be returned to the office.",
    "Year 9 science trip to the museum on Friday. Bring a packed lunch and your permission slip.",
    "Basketball trials for the senior team are on Wednesday after school in the gym.",
    "The canteen has a new menu: sushi on Tuesdays and pasta on Thursdays.",
    "Maths homework club meets every Thursday lunchtime in room B12. Everyone welcome.",
    "Lost property: a blue jacket and a calculator were found in the gym.",
    "Reminder: the swimming sports day is next Friday. Bring your togs and a towel.",
    "School photos will be taken on Tuesday morning. Wear your full uniform.",
]
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words="english")
notice_vectors = vectorizer.fit_transform(notices)

def search(query, top=3):
    scores = cosine_similarity(vectorizer.transform([query]), notice_vectors)[0]
    for i in scores.argsort()[::-1][:top]:
        print(f"{scores[i]:.2f}  {notices[i]}")

search("where do I return library books")

In [ ]:
search("I lost my jacket")
print()
search("when is swimming")

### 🧩 Exercise 3: Break the search engine
1. Search `"I misplaced my coat"`. Why does it fail even though notice 6 is exactly what you need? (TF-IDF only matches **the same words**; it doesn't know *coat* ≈ *jacket*. Next lesson fixes this with **word embeddings**!)
2. Add 3 notices of your own and test them.

### 🚀 Challenge
Use your own `preprocess()` function as the tokeniser: `TfidfVectorizer(tokenizer=preprocess, token_pattern=None)`. Now does `"returning books"` match `"returned"`? Why?

## 📌 Summary
- Preprocessing: **tokenise → lowercase → remove stop words → stem/lemmatise** (but think before removing words!)
- **Bag of Words** counts words, ignoring order (n-grams help a bit)
- **TF-IDF** rewards words that are frequent here but rare elsewhere
- **Cosine similarity** compares vectors → a simple search engine
- Weakness: no understanding of **meaning** or synonyms